# RMS Arrhythmia Detection — SHAP Explainability on MIT-BIH

This notebook evaluates the **saved, unchanged INT8 TensorFlow Lite RMS model** on MIT-BIH ECG beats and applies post-hoc SHAP explanations. It does not retrain the model or modify its weights.

**Important:** SHAP explanations describe how input features influence model outputs relative to a background dataset; they do not establish medical causality. The supplied training class order is `["N", "S", "V", "F", "Q"]`; verify it against the original training code before interpreting results. The notebook's initial split is beat-level stratified, not record-wise, so results may be optimistic. For publication-grade generalization results, use a record-wise split.

In [ ]:
%pip -q install wfdb shap scikit-learn pandas matplotlib

## 1. Upload model and MIT-BIH dataset ZIP

Upload `rms_arrhythmia_int8.tflite` and `mit-bih-arrhythmia-database-1.0.0.zip`. If already in Drive, mount Drive and set `MODEL_PATH` and `DATASET_ZIP` manually in the next cell.

In [ ]:
from google.colab import files
uploaded = files.upload()
print("Uploaded:", list(uploaded.keys()))

In [ ]:
from pathlib import Path
import zipfile, os
WORK_DIR = Path("/content/rms_xai")
WORK_DIR.mkdir(parents=True, exist_ok=True)

# Auto-detect uploaded files; edit manually if needed.
tflite_candidates = list(Path("/content").glob("*.tflite"))
zip_candidates = list(Path("/content").glob("*.zip"))
if not tflite_candidates:
    raise FileNotFoundError("Upload the saved .tflite model.")
if not zip_candidates:
    raise FileNotFoundError("Upload the MIT-BIH dataset ZIP.")
MODEL_PATH = tflite_candidates[0]
DATASET_ZIP = next((p for p in zip_candidates if "mit" in p.name.lower()), zip_candidates[0])
EXTRACT_DIR = WORK_DIR / "mit_bih"
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
if not any(EXTRACT_DIR.rglob("*.hea")):
    with zipfile.ZipFile(DATASET_ZIP, "r") as zf:
        zf.extractall(EXTRACT_DIR)
print("Model:", MODEL_PATH)
print("Dataset ZIP:", DATASET_ZIP)
print("Header files:", len(list(EXTRACT_DIR.rglob("*.hea"))))

## 2. Configure class mapping and preprocessing

This matches the supplied RMS training notebook. Do not change `CLASS_NAMES` unless you verify the saved model's output-index mapping.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import wfdb
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

CLASS_MAP = {
    "N": "N", "L": "N", "R": "N", "e": "N", "j": "N",
    "A": "S", "a": "S", "J": "S", "S": "S",
    "V": "V", "E": "V", "F": "F",
    "/": "Q", "f": "Q", "Q": "Q", "?": "Q", "|": "Q",
    "x": "Q", "u": "Q", "!": "Q"
}
CLASS_NAMES = ["N", "S", "V", "F", "Q"]
LABEL_TO_ID = {name: i for i, name in enumerate(CLASS_NAMES)}
WINDOW_LEFT, WINDOW_RIGHT = 90, 97
WINDOW_LEN = WINDOW_LEFT + WINDOW_RIGHT


## 3. Load the saved TFLite model and inspect quantization metadata

In [ ]:
import tensorflow as tf
interpreter = tf.lite.Interpreter(model_path=str(MODEL_PATH))
interpreter.allocate_tensors()
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()
print("Input:", input_details)
print("Output:", output_details)
if len(input_details) != 1 or len(output_details) != 1:
    raise ValueError("Expected a single-input, single-output model.")
inp, out = input_details[0], output_details[0]
print("Input shape/dtype/quantization:", inp["shape"], inp["dtype"], inp["quantization"])
print("Output shape/dtype/quantization:", out["shape"], out["dtype"], out["quantization"])


## 4. Extract and preprocess MIT-BIH beats

The notebook uses channel 0, extracts 187 samples (`sample-90` through `sample+96`), and applies per-window z-score normalization, matching the supplied training preprocessing.

In [ ]:
def find_record_stems(root):
    stems = []
    for hea in root.rglob("*.hea"):
        stem = hea.with_suffix("")
        if stem.with_suffix(".atr").exists():
            stems.append(stem)
    return sorted(set(stems), key=lambda p: str(p))

record_stems = find_record_stems(EXTRACT_DIR)
if not record_stems:
    raise FileNotFoundError("No .hea files with matching .atr annotations found in extracted dataset.")
print("Annotated records found:", len(record_stems))

X_list, y_list, record_list, symbol_list = [], [], [], []
skipped = {"unknown_symbol": 0, "invalid_window": 0, "read_error": 0}
for stem in record_stems:
    try:
        signals, fields = wfdb.rdsamp(str(stem))
        ann = wfdb.rdann(str(stem), "atr")
    except Exception as e:
        skipped["read_error"] += 1
        print(f"Skipping {stem.name}: {type(e).__name__}: {e}")
        continue
    ecg = signals[:, 0].astype(np.float32)
    for sample, symbol in zip(ann.sample, ann.symbol):
        cls = CLASS_MAP.get(symbol)
        if cls is None:
            skipped["unknown_symbol"] += 1
            continue
        start, end = int(sample)-WINDOW_LEFT, int(sample)+WINDOW_RIGHT
        if start < 0 or end > len(ecg):
            skipped["invalid_window"] += 1
            continue
        beat = ecg[start:end]
        if len(beat) != WINDOW_LEN or not np.all(np.isfinite(beat)):
            skipped["invalid_window"] += 1
            continue
        sd = float(np.std(beat))
        if sd < 1e-6:
            skipped["invalid_window"] += 1
            continue
        beat = (beat - float(np.mean(beat))) / sd
        X_list.append(beat.astype(np.float32))
        y_list.append(LABEL_TO_ID[cls])
        record_list.append(stem.name)
        symbol_list.append(symbol)

if not X_list:
    raise RuntimeError("No valid beats extracted; check dataset layout and annotations.")
X = np.stack(X_list).astype(np.float32)
y = np.asarray(y_list, dtype=np.int64)
record_ids = np.asarray(record_list)
symbols = np.asarray(symbol_list)
print("X shape:", X.shape, "y shape:", y.shape)
print("Skipped:", skipped)
print(pd.Series([CLASS_NAMES[i] for i in y]).value_counts().reindex(CLASS_NAMES, fill_value=0))

## 5. Split data and define TFLite prediction wrapper

The split below is a reproducible beat-level stratified split for a first run. It can leak record-specific characteristics across partitions, so do not present it as a patient-independent evaluation. The wrapper reads quantization parameters from the model metadata; it does not change model weights.

In [ ]:
X_train, X_test, y_train, y_test, rec_train, rec_test = train_test_split(
    X, y, record_ids, test_size=0.20, random_state=42, stratify=y
)
print("Train:", X_train.shape, "Test:", X_test.shape,
      "Distinct test records:", len(np.unique(rec_test)))

INPUT_SCALE, INPUT_ZERO_POINT = inp["quantization"]
OUTPUT_SCALE, OUTPUT_ZERO_POINT = out["quantization"]

def prepare_input_batch(x_batch):
    x_batch = np.asarray(x_batch, dtype=np.float32)
    if x_batch.ndim == 1:
        x_batch = x_batch[None, :]
    expected_shape = tuple(int(v) for v in inp["shape"])
    nonbatch_shape = expected_shape[1:]
    if int(np.prod(nonbatch_shape)) != x_batch.shape[1]:
        raise ValueError(f"Model expects {nonbatch_shape}, but each beat has {x_batch.shape[1]} samples.")
    if inp["dtype"] == np.int8:
        if not INPUT_SCALE:
            raise ValueError("Invalid input quantization scale.")
        q = np.rint(x_batch / INPUT_SCALE + INPUT_ZERO_POINT)
        q = np.clip(q, -128, 127).astype(np.int8)
    elif inp["dtype"] == np.uint8:
        q = np.rint(x_batch / INPUT_SCALE + INPUT_ZERO_POINT)
        q = np.clip(q, 0, 255).astype(np.uint8)
    else:
        q = x_batch.astype(inp["dtype"])
    return q.reshape((len(x_batch),) + nonbatch_shape)

def decode_output(raw):
    raw = np.asarray(raw)
    if out["dtype"] in (np.int8, np.uint8):
        scale, zero_point = out["quantization"]
        if scale and scale > 0:
            return (raw.astype(np.float32) - zero_point) * scale
    return raw.astype(np.float32)

def predict_scores(x_batch):
    batch = prepare_input_batch(x_batch)
    rows = []
    for i in range(len(batch)):
        interpreter.set_tensor(inp["index"], batch[i:i+1])
        interpreter.invoke()
        rows.append(decode_output(interpreter.get_tensor(out["index"])).reshape(-1))
    return np.vstack(rows)

print("Sample output shape:", predict_scores(X_test[:2]).shape)
print("Configured output class order:", CLASS_NAMES)
print("Verify output-index mapping against original training code before reporting results.")

## 6. Evaluate the original model

In [ ]:
def predict_in_batches(x, batch_size=64):
    return np.vstack([predict_scores(x[i:i+batch_size]) for i in range(0, len(x), batch_size)])

y_score = predict_in_batches(X_test)
y_pred = np.argmax(y_score, axis=1)
print(f"Beat-level accuracy (random stratified split): {accuracy_score(y_test, y_pred):.4f}")
print(classification_report(y_test, y_pred, labels=np.arange(len(CLASS_NAMES)),
                            target_names=CLASS_NAMES, zero_division=0))
cm = confusion_matrix(y_test, y_pred, labels=np.arange(len(CLASS_NAMES)))
fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES).plot(ax=ax, cmap="Blues", values_format="d", colorbar=False)
ax.set_title("RMS TFLite — MIT-BIH beat-level test split")
plt.xticks(rotation=45); plt.tight_layout(); plt.show()

## 7. Select SHAP background and test examples

Kernel SHAP is model-agnostic but computationally expensive. Start with a small pilot: two background beats per class and up to five examples. Increase settings only after a successful run.

In [ ]:
import shap
rng = np.random.default_rng(42)
BACKGROUND_PER_CLASS = 2
N_EXPLAIN = 5
NSAMPLES = 100

background_indices = []
for class_id in range(len(CLASS_NAMES)):
    candidates = np.flatnonzero(y_train == class_id)
    if len(candidates):
        background_indices.extend(rng.choice(candidates, size=min(BACKGROUND_PER_CLASS, len(candidates)), replace=False).tolist())
X_background = X_train[background_indices]

selected = []
for class_id in range(len(CLASS_NAMES)):
    candidates = np.flatnonzero(y_pred == class_id)
    if len(candidates):
        selected.append(int(candidates[0]))
    if len(selected) >= N_EXPLAIN:
        break
if len(selected) < min(N_EXPLAIN, len(X_test)):
    for idx in rng.permutation(len(X_test)):
        if int(idx) not in selected:
            selected.append(int(idx))
        if len(selected) >= min(N_EXPLAIN, len(X_test)):
            break

X_explain = X_test[selected]
y_explain = y_test[selected]
pred_explain = y_pred[selected]
display(pd.DataFrame({
    "test_index": selected,
    "true_class": [CLASS_NAMES[i] for i in y_explain],
    "predicted_class": [CLASS_NAMES[i] for i in pred_explain],
    "max_output_score": np.max(y_score[selected], axis=1)
}))
print("Background:", X_background.shape, "Explained:", X_explain.shape)

## 8. Run SHAP KernelExplainer

Each output class is explained independently. SHAP values describe contribution to the model output relative to the selected background distribution. The 187 time samples are correlated, so interpret contiguous regions cautiously.

In [ ]:
explainer = shap.KernelExplainer(predict_scores, X_background)
shap_values_raw = explainer.shap_values(
    X_explain, nsamples=NSAMPLES, l1_reg="num_features(20)"
)
print("SHAP computation complete. Return type:", type(shap_values_raw))
if isinstance(shap_values_raw, list):
    print("Per-output shapes:", [np.asarray(v).shape for v in shap_values_raw])
else:
    print("SHAP array shape:", np.asarray(shap_values_raw).shape)

In [ ]:
def normalize_shap_values(values, n_samples, n_features, n_outputs):
    if isinstance(values, list):
        return [np.asarray(v) for v in values]
    arr = np.asarray(values)
    if arr.ndim == 3:
        if arr.shape == (n_samples, n_features, n_outputs):
            return [arr[:, :, k] for k in range(n_outputs)]
        if arr.shape == (n_outputs, n_samples, n_features):
            return [arr[k, :, :] for k in range(n_outputs)]
    raise ValueError(f"Unrecognized SHAP shape {arr.shape}; inspect the printed shapes.")

shap_by_class = normalize_shap_values(shap_values_raw, len(X_explain), X_explain.shape[1], len(CLASS_NAMES))
print("Normalized SHAP shapes:", [v.shape for v in shap_by_class])

## 9. Plot ECG waveforms and SHAP attributions

Positive SHAP values increase the selected output relative to the explainer baseline; negative values decrease it. These are model output contributions, not clinical risk values.

In [ ]:
def plot_beat_shap(example_pos, class_id=None):
    x = X_explain[example_pos]
    true_id, pred_id = int(y_explain[example_pos]), int(pred_explain[example_pos])
    if class_id is None:
        class_id = pred_id
    sv = shap_by_class[class_id][example_pos]
    fig, axes = plt.subplots(2, 1, figsize=(13, 5), sharex=True,
                             gridspec_kw={"height_ratios": [2, 1]})
    axes[0].plot(np.arange(WINDOW_LEN), x, linewidth=1.4)
    axes[0].axvline(WINDOW_LEFT, linestyle="--", linewidth=1, label="Annotated beat location")
    axes[0].set_ylabel("Normalized ECG")
    axes[0].set_title(f"Example {example_pos}: true={CLASS_NAMES[true_id]}, predicted={CLASS_NAMES[pred_id]}, explained={CLASS_NAMES[class_id]}")
    axes[0].legend(loc="upper right")
    axes[1].bar(np.arange(WINDOW_LEN), sv, width=1.0)
    axes[1].axhline(0, linewidth=0.8)
    axes[1].axvline(WINDOW_LEFT, linestyle="--", linewidth=1)
    axes[1].set_xlabel("Sample index in 187-sample window")
    axes[1].set_ylabel("SHAP value")
    plt.tight_layout(); plt.show()

for i in range(len(X_explain)):
    plot_beat_shap(i)

## 10. Global attribution summary

In [ ]:
global_importance = np.mean(np.stack([np.abs(v) for v in shap_by_class], axis=0), axis=(0, 1))
top_n = 30
top_idx = np.argsort(global_importance)[-top_n:]
plt.figure(figsize=(12, 5))
plt.bar(top_idx, global_importance[top_idx])
plt.xlabel("Sample index in ECG window")
plt.ylabel("Mean absolute SHAP value")
plt.title(f"Global sample attribution (top {top_n} positions; selected examples only)")
plt.tight_layout(); plt.show()

importance_df = pd.DataFrame({"sample_index": np.arange(WINDOW_LEN),
                              "mean_absolute_shap": global_importance}).sort_values("mean_absolute_shap", ascending=False)
display(importance_df.head(20))

## 11. Optional SHAP summary plot for one output class

In [ ]:
CLASS_TO_EXPLAIN = "V"  # change only after confirming output-index mapping
class_id = CLASS_NAMES.index(CLASS_TO_EXPLAIN)
shap.summary_plot(shap_by_class[class_id], X_explain,
                  feature_names=[f"sample_{i}" for i in range(WINDOW_LEN)],
                  max_display=20, show=True)

## 12. Export predictions and SHAP values

In [ ]:
results_df = pd.DataFrame({
    "record_id": rec_test,
    "true_class": [CLASS_NAMES[i] for i in y_test],
    "predicted_class": [CLASS_NAMES[i] for i in y_pred],
    "correct": (y_test == y_pred)
})
results_csv = WORK_DIR / "rms_mitbih_test_predictions.csv"
results_df.to_csv(results_csv, index=False)

shap_array = np.stack(shap_by_class, axis=0)  # outputs x explained samples x features
shap_npz = WORK_DIR / "rms_shap_values.npz"
np.savez_compressed(shap_npz, shap_values=shap_array, X_explain=X_explain,
                    y_true=y_explain, y_pred=pred_explain,
                    class_names=np.asarray(CLASS_NAMES),
                    selected_test_indices=np.asarray(selected))
print("Saved:", results_csv, shap_npz)
from google.colab import files
files.download(str(results_csv))
files.download(str(shap_npz))

## 13. Research-paper reporting checklist

Before reporting results:
1. Verify the model output-index-to-class mapping from the original training code.
2. Use a record-wise/patient-wise split for final generalization metrics; the initial split here is beat-level.
3. Report valid beat counts and class distribution after filtering.
4. Report SHAP configuration (KernelExplainer, background size, explained sample count, `nsamples`, and regularization).
5. Include representative correctly classified and misclassified beats.
6. Explain that SHAP is post-hoc attribution relative to a chosen background; it does not prove physiological causality or clinical reliability.

The model weights and saved TFLite artifact are not modified by this notebook.